In [19]:
import pandas as pd
import numpy as np
import importlib
import statsmodels.formula.api as smf
from statsmodels.genmod.families import Gaussian
from statsmodels.genmod.cov_struct import Autoregressive, Unstructured

import analysis
importlib.reload(analysis)

<module 'analysis' from 'c:\\Users\\JoleneWium\\Documents\\Projects\\biostats\\PPH7095F\\20260402\\code\\analysis.py'>

In [20]:
df = analysis.load_data()

In [21]:
dat = df[
    [
        "id",
        "obs_time",
        "satisfaction",
        "sex",
        "age_marriage",
        "cohab",
        "income",
        "hw_all",
    ]
].dropna().copy()

dat = dat.sort_values(["id", "obs_time"]).reset_index(drop=True)

# visit order within person
dat["time_index"] = dat.groupby("id").cumcount().astype(int)

formula = (
    "satisfaction ~ time_index + sex + age_marriage + "
    "cohab + income + hw_all"
)

In [22]:
dat["time_index"] = dat.groupby("id").cumcount()
dat.groupby("id").size().describe()


count    494.000000
mean       7.718623
std        2.855509
min        3.000000
25%        6.000000
50%        8.000000
75%       10.000000
max       17.000000
dtype: float64

In [23]:
dat_un = dat[dat["time_index"] <= 3].copy()

In [29]:
# GEE 1: AR(1)
gee_ar1 = smf.gee(
    formula=formula,
    groups="id",
    time="time_index",
    data=dat_un,
    family=Gaussian(),
    cov_struct=Autoregressive(grid=True),
).fit()
gee_ar1.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                               GEE Regression Results                              
===================================================================================
Dep. Variable:                satisfaction   No. Observations:                 1935
Model:                                 GEE   No. clusters:                      494
Method:                        Generalized   Min. cluster size:                   3
                      Estimating Equations   Max. cluster size:                   4
Family:                           Gaussian   Mean cluster size:                 3.9
Dependence structure:       Autoregressive   Num. iterations:                     7
Date:                     Sun, 29 Mar 2026   Scale:                           2.455
Covariance type:                    robust   Time:                         11:36:10
================================================================================
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept        6.8799      0.403     17.052      0.000       6.089       7.671
sex[T.male]     -0.4900      0.156     -3.147      0.002      -0.795      -0.185
time_index      -0.4383      0.022    -20.015      0.000      -0.481      -0.395
age_marriage     0.0153      0.012      1.251      0.211      -0.009       0.039
cohab            0.3933      0.182      2.163      0.031       0.037       0.750
income       -1.233e-06   3.49e-06     -0.353      0.724   -8.07e-06     5.6e-06
hw_all          -0.5739      0.155     -3.697      0.000      -0.878      -0.270
==============================================================================
Skew:                         -0.0934   Kurtosis:                      -0.1285
Centered skew:                 0.0425   Centered kurtosis:              0.1728
==============================================================================
"""

In [28]:
# GEE 2: Unstructured
dat_un = dat[dat["time_index"] <= 3].copy()

gee_un = smf.gee(
    formula=formula,
    groups="id",
    time="time_index",
    data=dat_un,
    family=Gaussian(),
    cov_struct=Unstructured(),
).fit()
gee_un.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                               GEE Regression Results                              
===================================================================================
Dep. Variable:                satisfaction   No. Observations:                 1935
Model:                                 GEE   No. clusters:                      494
Method:                        Generalized   Min. cluster size:                   3
                      Estimating Equations   Max. cluster size:                   4
Family:                           Gaussian   Mean cluster size:                 3.9
Dependence structure:         Unstructured   Num. iterations:                     8
Date:                     Sun, 29 Mar 2026   Scale:                           2.454
Covariance type:                    robust   Time:                         11:34:54
================================================================================
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept        6.9301      0.392     17.694      0.000       6.162       7.698
sex[T.male]     -0.4832      0.154     -3.144      0.002      -0.784      -0.182
time_index      -0.4319      0.021    -20.279      0.000      -0.474      -0.390
age_marriage     0.0128      0.012      1.091      0.275      -0.010       0.036
cohab            0.3706      0.181      2.046      0.041       0.016       0.726
income       -6.237e-07   3.44e-06     -0.181      0.856   -7.37e-06    6.13e-06
hw_all          -0.5590      0.154     -3.637      0.000      -0.860      -0.258
==============================================================================
Skew:                         -0.0924   Kurtosis:                      -0.1353
Centered skew:                 0.0427   Centered kurtosis:              0.1787
==============================================================================
"""

In [31]:
analysis.table2_gee(df)

,Model,Variable,Estimate,SE,95% CI
0,GEE 1: AR(1),Intercept,6.972,0.412,6.165 to 7.779
1,GEE 1: AR(1),sex[T.male],-0.394,0.155,-0.697 to -0.091
2,GEE 1: AR(1),time_index,-0.295,0.009,-0.313 to -0.277
3,GEE 1: AR(1),age_marriage,0.010,0.012,-0.014 to 0.034
4,GEE 1: AR(1),cohab,0.311,0.183,-0.047 to 0.669
5,GEE 1: AR(1),income,-0.000,0.000,-0.0 to 0.0
6,GEE 1: AR(1),hw_all,-0.549,0.155,-0.853 to -0.246
7,GEE 2: Unstructured,Intercept,6.930,0.392,6.162 to 7.698
8,GEE 2: Unstructured,sex[T.male],-0.483,0.154,-0.784 to -0.182
9,GEE 2: Unstructured,time_index,-0.432,0.021,-0.474 to -0.39
